# 01 - Look at the prepared data (VS Code)
Run top to bottom **after** `audit_audio.py`, `build_manifest.py` and `extract_features.py`.
Choose the kernel **Python (smartlistener)** at the top right of VS Code.
Goal: see with your own eyes that the labels, splits and spectrograms make sense **before** any model is trained.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display

from smart_listener import audio
from smart_listener.config import config_fingerprint, load_config, resolve_path

cfg = load_config()
manifest = pd.read_csv(resolve_path(cfg, "manifest_dir") / "manifest.csv", keep_default_na=False)
FEAT = resolve_path(cfg, "features_dir")
MONO = resolve_path(cfg, "mono_dir")
meta = json.load(open(FEAT / "features_meta.json"))
assert meta["config_fingerprint"] == config_fingerprint(cfg), "Features were made with different settings. Run extract_features.py --force"
print("Settings fingerprint matches the saved features:", meta["config_fingerprint"])
GROUP = "pump_id_00"      # change to any: fan_id_02, pump_id_00, ...

## 1. How many files are in each split? (every machine)

In [ ]:
display(manifest.groupby(["group", "split_ad", "label_name"]).size().unstack([1, 2], fill_value=0))

**Check:** `train` has only `normal`. `val` and `test` have equal normal and abnormal counts. `buffer` files are thrown away on purpose
(they sit between the blocks so neighbouring recordings cannot leak across).

## 2. Average sound of normal vs abnormal (test split, one machine)

In [ ]:
g = manifest[manifest.group == GROUP].sort_values(["label_name", "index_in_label"]).reset_index(drop=True)
X = np.load(FEAT / "logmel" / f"{GROUP}.npy", mmap_mode="r")
test = g[g.split_ad == "test"]
mean_n = np.mean([X[i].mean(axis=1) for i in test.index[test.label == 0]], axis=0)
mean_a = np.mean([X[i].mean(axis=1) for i in test.index[test.label == 1]], axis=0)
mel_freqs = __import__("librosa").mel_frequencies(n_mels=cfg["features"]["n_mels"], fmin=cfg["features"]["fmin"], fmax=cfg["features"]["fmax"])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(mel_freqs, mean_n, label="normal"); ax[0].plot(mel_freqs, mean_a, label="abnormal")
ax[0].set_xlabel("frequency (Hz)"); ax[0].set_ylabel("mean level (dB)"); ax[0].legend(); ax[0].set_title(GROUP)
ax[1].bar(range(len(mean_n)), mean_a - mean_n); ax[1].set_xlabel("mel band"); ax[1].set_ylabel("abnormal minus normal (dB)")
ax[1].set_title("Where do abnormal sounds differ?")
plt.tight_layout(); plt.show()

## 3. Example spectrograms: one normal and one abnormal (same colour scale)

In [ ]:
i_n = test.index[test.label == 0][0]; i_a = test.index[test.label == 1][0]
vmin, vmax = float(min(X[i_n].min(), X[i_a].min())), float(max(X[i_n].max(), X[i_a].max()))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for a, i, name in ((ax[0], i_n, "normal"), (ax[1], i_a, "abnormal")):
    im = a.imshow(X[i], origin="lower", aspect="auto", vmin=vmin, vmax=vmax)
    a.set_title(f"{name}: {g.loc[i, 'file_id']}"); a.set_xlabel("frame (about 32 ms each)")
ax[0].set_ylabel("mel band"); fig.colorbar(im, ax=ax, label="dB")
plt.show()

## 4. Listen (use headphones; the abnormal clip may sound only slightly different)

In [ ]:
for i in (i_n, i_a):
    print(g.loc[i, "label_name"], g.loc[i, "file_id"])
    display(Audio(str(MONO / g.loc[i, "path"])))

## 5. Normalisation check
The saved mean and std must come only from this machine's **training** files.

In [ ]:
st = json.load(open(FEAT / "norm" / f"{GROUP}_anomaly.json"))
n_train = int((g.split_ad == "train").sum())
print("files used to fit the statistics:", st["n_files"], "| training files in manifest:", n_train)
assert st["n_files"] == n_train, "Statistics were not fitted on exactly the training files!"
z = audio.apply_norm(np.asarray(X[g.index[g.split_ad == "train"][0]]), np.array(st["mean"]), np.array(st["std"]))
print("normalised training clip: mean %.2f, std %.2f (one clip will not be exactly 0 and 1, but should be close)" % (z.mean(), z.std()))